# LangGraph Assistant Walkthrough

Step-by-step lab for the ETL Observability **Incident RCA / DQ** agents.

You will test each layer:

1. Tool registry (`TOOLS`)
2. Single Metadata tool call (FakeMeta — offline)
3. LangChain `StructuredTool` wrappers
4. Scripted ReAct loop (no OpenRouter) — tool → result → answer
5. Full `run_langgraph_turn`
6. Fact-checking / grounding
7. Optional: live Metadata + OpenRouter

**Setup** (PowerShell):

```powershell
cd "D:\etl pipeline"
pip install -e packages/assistants
pip install jupyter ipykernel
cd packages/assistants/notebooks
jupyter notebook langgraph_assistant_walkthrough.ipynb
```

Run cells top → bottom. Offline cells need **no** Metadata server and **no** API key.

## 1. Imports + path setup

In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path
from typing import Any

ROOT = Path.cwd().resolve()
for cand in [ROOT, *ROOT.parents]:
    src = cand / "src"
    if (src / "assistants").is_dir():
        sys.path.insert(0, str(src))
        print("added:", src)
        break
else:
    src = (Path("..") / "src").resolve()
    sys.path.insert(0, str(src))
    print("added:", src)

from assistants.agentic.tools import TOOLS, run_tool_calls, build_allowed_ids
from assistants.agentic.langgraph_agent import build_metadata_tools, run_langgraph_turn
from assistants.shared.chat import fact_check_reply, clean_reply
from assistants.config import OPENROUTER_API_KEY, OPENROUTER_MODEL, METADATA_API_BASE

print("OPENROUTER configured:", bool(OPENROUTER_API_KEY))
print("MODEL:", OPENROUTER_MODEL)
print("METADATA_API_BASE:", METADATA_API_BASE)
print("tool count:", len(TOOLS))

added: D:\etl pipeline\packages\assistants\src


C:\Users\Vithi\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


OPENROUTER configured: True
MODEL: openrouter/free
METADATA_API_BASE: http://127.0.0.1:8000
tool count: 10


## 2. Fake Metadata client (offline stand-in)

Same shape as production `MetadataClient`, but in-memory — so you can see tool I/O without starting `:8000`.

In [5]:
class FakeMeta:
    """In-memory Metadata stand-in for notebook demos."""

    def get_incident(self, tenant_id: str, incident_key: str) -> dict[str, Any]:
        return {
            "incident_key": incident_key,
            "title": "Pipeline failed: finance_etl",
            "status": "open",
            "severity": "high",
            "root_asset_type": "pipeline",
            "root_asset_id": "finance_etl",
            "summary": "Connection timeout to Snowflake",
            "error_message": "Connection timeout to Snowflake",
        }

    def list_alerts(self, tenant_id: str, *, asset_id: str | None = None, limit: int = 200):
        return [{
            "alert_key": "alert:run-1",
            "title": "Pipeline failed: finance_etl",
            "status": "open",
            "severity": "high",
            "asset_id": "finance_etl",
            "message": "Connection timeout to Snowflake",
            "monitor_type": "pipeline_failure",
        }]

    def list_executions(self, tenant_id: str, pipeline_id: str | None = None, limit: int = 100):
        return [{
            "execution_id": "manual__2026-07-28",
            "pipeline_id": "finance_etl",
            "task_id": "extract_orders",
            "status": "failed",
            "error_message": "Connection timeout to Snowflake",
            "source_tool": "airflow",
            "started_at": "2026-07-28T10:00:00Z",
            "deep_link": "https://airflow.example.com/dags/finance_etl/grid?dag_run_id=manual__2026-07-28",
            "deep_link_label": "Open in Airflow",
        }]

    def get_pipeline_dashboard(self, tenant_id: str, pipeline_id: str):
        return {
            "pipeline": {"pipeline_id": pipeline_id, "source_tool": "airflow", "status": "failed"},
            "metrics": {"failed": 1, "succeeded": 0, "total_runs": 1},
            "related_datasets": ["ANALYTICS.RAW.ORDERS", "ANALYTICS.MART.FCT_ORDERS"],
            "pipeline_io": [{
                "upstream_dataset_id": "ANALYTICS.RAW.ORDERS",
                "downstream_dataset_id": "ANALYTICS.MART.FCT_ORDERS",
                "source_tool": "airflow",
            }],
            "executions": self.list_executions(tenant_id, pipeline_id),
        }

    def get_dataset(self, tenant_id: str, dataset_id: str):
        return {"dataset_id": dataset_id, "name": dataset_id.split(".")[-1], "platform": "snowflake", "row_count": 50}

    def get_blast_radius(self, tenant_id: str, dataset_id: str):
        return {"dataset_id": dataset_id, "downstream": [], "count": 0}

    def list_lineage(self, tenant_id: str, dataset_id: str | None = None, limit: int = 200):
        return [{
            "upstream_dataset_id": "ANALYTICS.RAW.ORDERS",
            "downstream_dataset_id": "ANALYTICS.MART.FCT_ORDERS",
            "transform": "finance_etl",
        }]

    def list_monitors(self, tenant_id: str, limit: int = 200):
        return [{
            "monitor_key": "mon:demo:volume:ANALYTICS.MART.FCT_ORDERS",
            "monitor_type": "volume",
            "asset_id": "ANALYTICS.MART.FCT_ORDERS",
            "enabled": True,
        }]

    def list_check_results(self, tenant_id: str, *, asset_id=None, monitor_type=None, limit=100):
        return [{
            "id": 1,
            "monitor_type": "volume",
            "asset_id": asset_id or "ANALYTICS.MART.FCT_ORDERS",
            "status": "anomalous",
            "metric_value": 50,
            "details": {"row_count": 50, "expected_min": 10000},
        }]

    def list_metrics(self, tenant_id: str, *, asset_id=None, name=None, limit=100):
        rows = [
            {"name": "row_count", "asset_id": "ANALYTICS.MART.FCT_ORDERS", "value": 50, "recorded_at": "2026-07-28T10:00:00Z"},
            {"name": "freshness_lag_hours", "asset_id": "ANALYTICS.MART.FCT_ORDERS", "value": 6, "recorded_at": "2026-07-28T10:00:00Z"},
        ]
        if asset_id:
            rows = [r for r in rows if r["asset_id"] == asset_id]
        if name:
            rows = [r for r in rows if r["name"] == name]
        return rows[:limit]

meta = FakeMeta()
TENANT = "demo"
INCIDENT_KEY = "inc:demo:pipeline:finance_etl:pipeline_failure"
BOUND = {
    "incident_key": INCIDENT_KEY,
    "pipeline_id": "finance_etl",
    "dataset_id": "ANALYTICS.MART.FCT_ORDERS",
}
print("bound context:")
print(json.dumps(BOUND, indent=2))

bound context:
{
  "incident_key": "inc:demo:pipeline:finance_etl:pipeline_failure",
  "pipeline_id": "finance_etl",
  "dataset_id": "ANALYTICS.MART.FCT_ORDERS"
}


## 3. Inspect the tool registry

Every LangGraph tool is defined in `TOOLS` — name, description, args, and a function that calls Metadata.

In [6]:
rows = []
for name, spec in TOOLS.items():
    rows.append({
        "tool": name,
        "description": spec["description"],
        "required": ", ".join(spec.get("required") or []) or "—",
        "args": ", ".join((spec.get("properties") or {}).keys()) or "—",
    })

try:
    import pandas as pd
    display(pd.DataFrame(rows))
except Exception:
    for r in rows:
        print(f"{r['tool']:28} | {r['description']}")
        print(f"{'':28} | required={r['required']}  args={r['args']}\n")

,tool,description,required,args
0,get_incident,Fetch one incident by key,incident_key,incident_key
1,list_executions,List pipeline/task run history,—,"pipeline_id, limit"
2,get_pipeline_dashboard,"Pipeline metrics, IO, related datasets, recent...",pipeline_id,pipeline_id
3,list_check_results,Data quality check results for an asset,—,"asset_id, monitor_type, limit"
4,list_monitors,Monitors configured for an asset,—,"asset_id, limit"
5,list_alerts,Alerts for an asset,—,"asset_id, limit"
6,get_blast_radius,Downstream datasets affected by a dataset issue,dataset_id,dataset_id
7,list_lineage,Upstream/downstream lineage edges,—,"dataset_id, limit"
8,get_dataset,Dataset catalog details,dataset_id,dataset_id
9,list_metrics,"Time-series metrics for charts (row_count, lag...",—,"asset_id, name, limit"


## 4. Call tools directly (no LLM yet)

`run_tool_calls` is what LangGraph tools wrap. Run a few yourself and inspect the evidence bag + `tool_trace`.

In [7]:
evidence = run_tool_calls(
    meta,  # type: ignore[arg-type]
    TENANT,
    [
        ("get_incident", {"incident_key": INCIDENT_KEY}),
        ("list_executions", {"pipeline_id": "finance_etl"}),
        ("get_pipeline_dashboard", {"pipeline_id": "finance_etl"}),
    ],
)

print("=== tool_trace ===")
print(json.dumps(evidence["tool_trace"], indent=2))

print("\n=== incident title ===")
print(evidence["incident"]["title"])
print("error:", evidence["incident"]["error_message"])

print("\n=== failed execution ===")
ex = evidence["executions"][0]
print(ex["task_id"], ex["status"], "→", ex["error_message"])
print("deep_link:", ex["deep_link"])

print("\n=== related datasets ===")
print(evidence["pipeline_dashboard"]["related_datasets"])

=== tool_trace ===
[
  {
    "tool": "get_incident",
    "ok": true,
    "args": {
      "incident_key": "inc:demo:pipeline:finance_etl:pipeline_failure"
    }
  },
  {
    "tool": "list_executions",
    "ok": true,
    "args": {
      "pipeline_id": "finance_etl"
    }
  },
  {
    "tool": "get_pipeline_dashboard",
    "ok": true,
    "args": {
      "pipeline_id": "finance_etl"
    }
  }
]

=== incident title ===
Pipeline failed: finance_etl
error: Connection timeout to Snowflake

=== failed execution ===
extract_orders failed → Connection timeout to Snowflake
deep_link: https://airflow.example.com/dags/finance_etl/grid?dag_run_id=manual__2026-07-28

=== related datasets ===
['ANALYTICS.RAW.ORDERS', 'ANALYTICS.MART.FCT_ORDERS']


In [8]:
# DQ-side tools
dq = run_tool_calls(
    meta,  # type: ignore[arg-type]
    TENANT,
    [
        ("get_dataset", {"dataset_id": "ANALYTICS.MART.FCT_ORDERS"}),
        ("list_check_results", {"asset_id": "ANALYTICS.MART.FCT_ORDERS"}),
        ("list_lineage", {"dataset_id": "ANALYTICS.MART.FCT_ORDERS"}),
        ("get_blast_radius", {"dataset_id": "ANALYTICS.MART.FCT_ORDERS"}),
        ("list_metrics", {"asset_id": "ANALYTICS.MART.FCT_ORDERS", "name": "row_count"}),
    ],
)
print("check status:", dq["check_results"][0]["status"], "metric=", dq["check_results"][0]["metric_value"])
print("lineage:", dq["lineage_edges"][0])
print("metrics:", dq["metrics"])
print("tool_trace ok?", all(t.get("ok") for t in dq["tool_trace"]))

check status: anomalous metric= 50
lineage: {'upstream_dataset_id': 'ANALYTICS.RAW.ORDERS', 'downstream_dataset_id': 'ANALYTICS.MART.FCT_ORDERS', 'transform': 'finance_etl'}
metrics: [{'name': 'row_count', 'asset_id': 'ANALYTICS.MART.FCT_ORDERS', 'value': 50, 'recorded_at': '2026-07-28T10:00:00Z'}]
tool_trace ok? True


## 5. Build LangChain StructuredTools

`build_metadata_tools` turns each registry entry into a LangChain tool the LangGraph agent can call.
Invoking a tool also **accumulates evidence** for later fact-checking.

In [9]:
evidence_bag: dict[str, Any] = {"tool_trace": []}
lc_tools = build_metadata_tools(meta, TENANT, BOUND, evidence_bag)  # type: ignore[arg-type]

print("LangChain tools:", [t.name for t in lc_tools])

# Pick one tool and call it like the agent would
get_incident = next(t for t in lc_tools if t.name == "get_incident")
raw = get_incident.invoke({"incident_key": INCIDENT_KEY})
print("\n--- get_incident raw JSON returned to the model ---")
print(raw[:500], "..." if len(raw) > 500 else "")

list_exec = next(t for t in lc_tools if t.name == "list_executions")
_ = list_exec.invoke({"pipeline_id": "finance_etl"})


print("\n--- evidence_bag keys after 2 tool calls ---")
print(sorted(k for k in evidence_bag if k != "tool_trace"))
print("tool_trace:", evidence_bag["tool_trace"])

LangChain tools: ['get_incident', 'list_executions', 'get_pipeline_dashboard', 'list_check_results', 'list_monitors', 'list_alerts', 'get_blast_radius', 'list_lineage', 'get_dataset', 'list_metrics']

--- get_incident raw JSON returned to the model ---
{"incident_key": "inc:demo:pipeline:finance_etl:pipeline_failure", "title": "Pipeline failed: finance_etl", "status": "open", "severity": "high", "root_asset_type": "pipeline", "root_asset_id": "finance_etl", "summary": "Connection timeout to Snowflake", "error_message": "Connection timeout to Snowflake"} 

--- evidence_bag keys after 2 tool calls ---
['executions', 'incident']
tool_trace: [{'tool': 'get_incident', 'ok': True, 'args': {'incident_key': 'inc:demo:pipeline:finance_etl:pipeline_failure'}}, {'tool': 'list_executions', 'ok': True, 'args': {'pipeline_id': 'finance_etl'}}]


## 6. Scripted chat model (simulate ReAct without OpenRouter)

This fake model returns a **tool-call message**, then a **final answer** — so you can watch LangGraph’s loop without spending tokens.

In [10]:
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.messages import AIMessage
from langchain_core.outputs import ChatGeneration, ChatResult
from pydantic import Field


class SequencedChatModel(BaseChatModel):
    """Returns scripted AIMessages; supports bind_tools used by create_agent."""

    responses: list[AIMessage] = Field(default_factory=list)
    call_i: int = 0

    @property
    def _llm_type(self) -> str:
        return "sequenced-fake"

    def bind_tools(self, tools: Any, **kwargs: Any) -> "SequencedChatModel":
        return self

    def _generate(self, messages, stop=None, run_manager=None, **kwargs):
        idx = min(self.call_i, len(self.responses) - 1)
        self.call_i += 1
        print(f"\n[model step {self.call_i}] messages so far: {len(messages)}")
        for m in messages[-3:]:
            role = type(m).__name__
            preview = str(getattr(m, "content", ""))[:80].replace("\n", " ")
            tcs = getattr(m, "tool_calls", None)
            print(f"  {role}: {preview!r} tool_calls={bool(tcs)}")
        return ChatResult(generations=[ChatGeneration(message=self.responses[idx])])


scripted = SequencedChatModel(
    responses=[
        # Step A: model decides to call tools
        AIMessage(
            content="",
            tool_calls=[
                {
                    "name": "get_incident",
                    "args": {"incident_key": INCIDENT_KEY},
                    "id": "call_1",
                    "type": "tool_call",
                },
                {
                    "name": "list_executions",
                    "args": {"pipeline_id": "finance_etl"},
                    "id": "call_2",
                    "type": "tool_call",
                },
            ],
        ),
        # Step B: after tool results are fed back, model answers
        AIMessage(
            content=(
                "finance_etl failed on extract_orders due to Connection timeout to Snowflake. "
                "Open the Airflow deep link for the full task log."
            )
        ),
    ]
)
print("scripted model ready")

scripted model ready


## 7. Run full LangGraph turn (offline)

This is the same path production chat uses (`run_langgraph_turn` → `create_agent`).

In [11]:
result = run_langgraph_turn(
    client=meta,  # type: ignore[arg-type]
    tenant_id=TENANT,
    question="What failed and why?",
    kind="incident_rca",
    bound=BOUND,
    chat_model=scripted,
)

print("agent_mode:", result["agent_mode"])
print("used_tools:", result["used_tools"])
print("grounded:", result["grounded"])
print("\n=== reply ===")
print(result["reply"])
print("\n=== tool_trace ===")
print(json.dumps(result["tool_trace"], indent=2))
print("\n=== allowed_citation_ids (sample) ===")
print(result["evidence"].get("allowed_citation_ids", [])[:12])


[model step 1] messages so far: 2
  SystemMessage: 'You are an ETL observability agent with Metadata tools (LangGraph / LangChain cr' tool_calls=False
  HumanMessage: 'What failed and why?' tool_calls=False

[model step 2] messages so far: 5
  AIMessage: '' tool_calls=True
  ToolMessage: '{"incident_key": "inc:demo:pipeline:finance_etl:pipeline_failure", "title": "Pip' tool_calls=False
  ToolMessage: '[{"execution_id": "manual__2026-07-28", "pipeline_id": "finance_etl", "task_id":' tool_calls=False
agent_mode: langgraph
used_tools: True
grounded: True

=== reply ===
finance_etl failed on extract_orders due to Connection timeout to Snowflake. Open the Airflow deep link for the full task log.

=== tool_trace ===
[
  {
    "tool": "get_incident",
    "ok": true,
    "args": {
      "incident_key": "inc:demo:pipeline:finance_etl:pipeline_failure"
    }
  },
  {
    "tool": "list_executions",
    "ok": true,
    "args": {
      "pipeline_id": "finance_etl"
    }
  }
]

=== allowed_citation

## 8. Fact-check / grounding

After the agent answers, we soft-check that claimed assets exist in evidence.

In [15]:
ev = result["evidence"]

good = "finance_etl failed on extract_orders. ANALYTICS.MART.FCT_ORDERS may be stale."
bad = "The issue is in FAKE.SCHEMA.INVENTED_TABLE which never appears in metadata."

for label, text in [("known assets", good), ("invented FQN", bad)]:
    reply, grounded, invented = fact_check_reply(text, ev)
    print(f"--- {label} ---")
    print("grounded:", grounded, " invented:", invented)
    print(reply)
    print()

print("clean_reply strips internal ids:")
print(clean_reply("See alert:abc-123 and check:99 for details."))

--- known assets ---
grounded: False  invented: ['ANALYTICS.MART.FCT_ORDERS']
finance_etl failed on extract_orders. ANALYTICS.MART.FCT_ORDERS may be stale.

_Note: metadata doesn't confirm these names from evidence: ANALYTICS.MART.FCT_ORDERS. Treat them as unverified._

--- invented FQN ---
grounded: False  invented: ['FAKE.SCHEMA.INVENTED_TABLE']
The issue is in FAKE.SCHEMA.INVENTED_TABLE which never appears in metadata.

_Note: metadata doesn't confirm these names from evidence: FAKE.SCHEMA.INVENTED_TABLE. Treat them as unverified._

clean_reply strips internal ids:
See  and  for details.


## 9. DQ walkthrough (scripted)

Same agent, different bound context + tools the model chooses.

In [16]:
dq_model = SequencedChatModel(
    responses=[
        AIMessage(
            content="",
            tool_calls=[
                {"name": "get_dataset", "args": {"dataset_id": "ANALYTICS.MART.FCT_ORDERS"}, "id": "d1", "type": "tool_call"},
                {"name": "list_check_results", "args": {"asset_id": "ANALYTICS.MART.FCT_ORDERS"}, "id": "d2", "type": "tool_call"},
                {"name": "list_lineage", "args": {"dataset_id": "ANALYTICS.MART.FCT_ORDERS"}, "id": "d3", "type": "tool_call"},
            ],
        ),
        AIMessage(
            content=(
                "Volume looks low on ANALYTICS.MART.FCT_ORDERS (about 50 rows). "
                "It is built from ANALYTICS.RAW.ORDERS via finance_etl."
            )
        ),
    ]
)

dq_result = run_langgraph_turn(
    client=meta,  # type: ignore[arg-type]
    tenant_id=TENANT,
    question="Which checks failed and what is upstream?",
    kind="dq_lineage",
    bound={"dataset_id": "ANALYTICS.MART.FCT_ORDERS"},
    chat_model=dq_model,
)

print(dq_result["reply"])
print("\ntools used:", [t["tool"] for t in dq_result["tool_trace"] if t.get("ok")])


[model step 1] messages so far: 2
  SystemMessage: 'You are an ETL observability agent with Metadata tools (LangGraph / LangChain cr' tool_calls=False
  HumanMessage: 'Which checks failed and what is upstream?' tool_calls=False

[model step 2] messages so far: 6
  ToolMessage: '{"dataset_id": "ANALYTICS.MART.FCT_ORDERS", "name": "FCT_ORDERS", "platform": "s' tool_calls=False
  ToolMessage: '[{"id": 1, "monitor_type": "volume", "asset_id": "ANALYTICS.MART.FCT_ORDERS", "s' tool_calls=False
  ToolMessage: '[{"upstream_dataset_id": "ANALYTICS.RAW.ORDERS", "downstream_dataset_id": "ANALY' tool_calls=False
Volume looks low on ANALYTICS.MART.FCT_ORDERS (about 50 rows). It is built from ANALYTICS.RAW.ORDERS via finance_etl.

tools used: ['get_dataset', 'list_check_results', 'list_lineage']


## 10. Optional — live Metadata + OpenRouter

Requires:
- Metadata API on `METADATA_API_BASE` (default `http://127.0.0.1:8000`)
- `OPENROUTER_API_KEY` in repo `.env`

Set `RUN_LIVE = True` only when ready (uses real tokens).

In [17]:
RUN_LIVE = False  # flip to True to hit real Metadata + OpenRouter

if not RUN_LIVE:
    print("Skipped live run. Set RUN_LIVE = True after Metadata + OpenRouter are ready.")
else:
    from assistants.metadata_client import MetadataClient

    live = MetadataClient()
    # Pick a real incident from your tenant, or leave and expect 404
    live_incident = INCIDENT_KEY
    try:
        inc = live.get_incident(TENANT, live_incident)
        print("loaded incident:", inc.get("title"))
    except Exception as e:
        print("Could not load incident — list one from UI / twin first:", e)
        raise

    live_result = run_langgraph_turn(
        client=live,
        tenant_id=TENANT,
        question="What failed and why? Summarize blast radius if known.",
        kind="incident_rca",
        bound={
            "incident_key": live_incident,
            "pipeline_id": inc.get("root_asset_id") if (inc.get("root_asset_type") or "").lower() == "pipeline" else None,
        },
        # chat_model omitted → real OpenRouter ChatOpenAI
    )
    print("agent_mode:", live_result["agent_mode"])
    print("used_tools:", live_result["used_tools"])
    print("tool_trace:", live_result["tool_trace"])
    print("\n=== live reply ===")
    print(live_result["reply"])

Skipped live run. Set RUN_LIVE = True after Metadata + OpenRouter are ready.


## Mental model (recap)

```text
User question
    → bound context (incident_key / dataset_id)
    → LangGraph create_agent(llm, StructuredTools)
    → model may emit tool_calls
    → each tool → Metadata HTTP (or FakeMeta)
    → results appended to messages + evidence bag
    → model final text
    → fact_check_reply → user-safe answer
```

Production entrypoints:

- Chat: `POST /v1/chat/sessions` / `POST /v1/dq/chat/sessions`
- A2A: `POST /a2a/jsonrpc`
- Core: `assistants.agentic.langgraph_agent.run_langgraph_turn`